# 05 — Sequential Models: Compact GRU & Conv1D

Trains lightweight recurrent and convolutional architectures (hidden dim ≤ 16) on PORTEX advisory sequences. Compact GRU achieves AUC 0.797, enabling real-time temporal risk estimation.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.models.sequential import build_compact_gru, build_compact_conv1d, build_sequences
from src.data.preprocessing import FEATURE_COLS, GROUP_COL, LABEL_COL
from src.utils.metrics import compute_all_metrics

df = pd.read_parquet("../data/processed/portex_labeled.parquet")

SEQ_LEN = 10


In [ ]:
# ── Build sequence arrays ────────────────────────────────
X_seq, y_seq, groups_seq = build_sequences(
    df, feature_cols=FEATURE_COLS, label_col=LABEL_COL,
    group_col=GROUP_COL, seq_len=SEQ_LEN
)
print(f"Sequence shapes: X={X_seq.shape}, y={y_seq.shape}")
print(f"Positive rate: {y_seq.mean()*100:.1f}%")


In [ ]:
# ── LOSO evaluation for GRU ─────────────────────────────
from sklearn.model_selection import LeaveOneGroupOut
import tensorflow as tf

logo = LeaveOneGroupOut()
gru_aucs = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X_seq, y_seq, groups_seq)):
    X_tr, X_te = X_seq[train_idx], X_seq[test_idx]
    y_tr, y_te = y_seq[train_idx], y_seq[test_idx]
    if y_te.sum() == 0:
        continue

    model_gru = build_compact_gru(SEQ_LEN, len(FEATURE_COLS))
    model_gru.fit(X_tr, y_tr, epochs=30, batch_size=32, verbose=0,
                  class_weight={0: 1, 1: int((y_tr==0).sum()/max(y_tr.sum(),1))})

    proba = model_gru.predict(X_te, verbose=0).flatten()
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(y_te, proba)
    gru_aucs.append(auc)
    print(f"  Fold {fold:02d}  storm={groups_seq[test_idx[0]]:<25}  AUC={auc:.3f}")

print(f"\nCompact GRU  Mean AUC: {np.mean(gru_aucs):.4f} ± {np.std(gru_aucs):.4f}")


In [ ]:
# ── LOSO evaluation for Conv1D ───────────────────────────
conv_aucs = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X_seq, y_seq, groups_seq)):
    X_tr, X_te = X_seq[train_idx], X_seq[test_idx]
    y_tr, y_te = y_seq[train_idx], y_seq[test_idx]
    if y_te.sum() == 0:
        continue

    model_conv = build_compact_conv1d(SEQ_LEN, len(FEATURE_COLS))
    model_conv.fit(X_tr, y_tr, epochs=30, batch_size=32, verbose=0,
                   class_weight={0: 1, 1: int((y_tr==0).sum()/max(y_tr.sum(),1))})

    proba = model_conv.predict(X_te, verbose=0).flatten()
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(y_te, proba)
    conv_aucs.append(auc)

print(f"Compact Conv1D  Mean AUC: {np.mean(conv_aucs):.4f} ± {np.std(conv_aucs):.4f}")


In [ ]:
# ── Performance comparison chart ────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
results = {
    "Compact GRU":    {"mean": np.mean(gru_aucs),  "std": np.std(gru_aucs)},
    "Compact Conv1D": {"mean": np.mean(conv_aucs), "std": np.std(conv_aucs)},
}
names = list(results.keys())
means = [v["mean"] for v in results.values()]
stds  = [v["std"]  for v in results.values()]
ax.bar(names, means, yerr=stds, capsize=6, color=["#4CAF50", "#03A9F4"],
       alpha=0.85, edgecolor="white")
ax.set_ylim(0.5, 1.0); ax.set_ylabel("AUC (LOSO)")
ax.set_title("Sequential Model Performance", fontsize=12, fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../results/figures/05_sequential_performance.png", dpi=150, bbox_inches="tight")
plt.show()
